In [1]:
#1 Installed libraries that allow: importing a translation model, a tokenizer and an evaluation tool (BLEU).
!pip install transformers sentencepiece sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.4 MB/s eta 0:00:00


In [2]:
# Uploaded the csv file, which contains extracted ENG-SLO pairs from the tmx files.
from google.colab import files
uploaded = files.upload()

Saving dgt_all.csv to dgt_all.csv


In [3]:
# Converts the csv into a dataframe.
import pandas as pd

df = pd.read_csv("dgt_all.csv")

print(df.head())
print(len(df))

                                                  en  \
0                Decision of the EEA Joint Committee   
1                                of 13 December 2013   
2  amending Annex I (Veterinary and phytosanitary...   
3                           THE EEA JOINT COMMITTEE,   
4  Having regard to the Agreement on the European...   

                                                  sl  
0                          Sklep Skupnega odbora EGP  
1                            z dne 13. decembra 2013  
2  o spremembi Priloge I (Veterinarske in fitosan...  
3                              SKUPNI ODBOR EGP JE –  
4  ob upoštevanju Sporazuma o Evropskem gospodars...  
87735


In [4]:
# Extracts a sample of 500 (couldn't do 1000, cauz a later cell takes more than an hour to run and then crashes either way ); ) for the training set, and a sample of 200 for the test set (which will not be used now, but later).
train_df = df.sample(500, random_state=42)
test_df = df.drop(train_df.index).sample(200, random_state=42)

print(len(train_df), len(test_df))

500 200


In [5]:
# Imports tools for tokenization and the NLLB translation model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

fb_model = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(fb_model)
model = AutoModelForSeq2SeqLM.from_pretrained(fb_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [10]:
# Made the model run on GPU instead of CPU, otherwise it didn't want to run at all even for 500 samples.
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [11]:
# Defines the source and target languages for the model to translate.
tokenizer.src_lang = "eng_Latn"
target_lang = "slv_Latn"

In [12]:
# Makes the model translate more than just 1 sentence at a time (for speed increase).
def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang)
    )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [14]:
# Makes the model translate 8 sentences at once. The model translates the ENG sentences into SLO and appends that to the list PREDICTIONS, and to REFERENCES we append the
# already translated SLO sentences, this is our Gold Standard.
batch_size = 8

predictions = []
references = []

for i in range(0, len(train_df), batch_size):
    batch = train_df.iloc[i:i+batch_size]

    en_texts = batch["en"].tolist()
    sl_texts = batch["sl"].tolist()

    preds = translate_batch(en_texts)

    predictions.extend(preds)
    references.extend(sl_texts)

In [15]:
# Just checking if everything works by printing 20 examples: the Model translation and how it compares to the Gold Standard.
for pred, ref in list(zip(predictions, references))[:20]:
  print('Model translation:', pred)
  print('Gold standard:', ref)
  print('-' * 40)

Model translation: Brez poseganja v člen 30 nobena pogodbenica ne sme naložiti znatnih kazni za manjše kršitve carinskih predpisov ali postopkovnih zahtev.
Gold standard: Brez poseganja v člen 30 nobena pogodbenica ne uvede velikih kazni za manjše kršitve carinskih predpisov ali postopkovnih zahtev.
----------------------------------------
Model translation: Tapered roller bearing, vključno s konom in tapered roller skupine:
Gold standard: Stožčasti ležaji, vključno sestavi iz notranjega obroča in kletke s stožčastimi valjčki:
----------------------------------------
Model translation: Združbeni svet EU-Liban podpira to obnovljeno partnerstvo kot paradigmo za novo, prilagojeno dvostransko sodelovanje, ki vključuje posodobljen politični dialog.
Gold standard: Pridružitveni svet EU-Libanon potrjuje to prenovljeno partnerstvo kot paradigmo za novo prilagojeno dvostransko sodelovanje, ki vključuje nadgraditev političnega dialoga.
----------------------------------------
Model translation: 

In [19]:
# Evaluating the model's translation compared to the Gold standard.
import sacrebleu

bleu = sacrebleu.corpus_bleu(predictions, [references])

print(f'BLEU score:, {bleu.score:.2f}')

BLEU score:, 36.17
